# Phase 2 — Notebook 05: Storage

**Goal:** Insert documents, sections, chunks (with embeddings), assets, and relationships into Postgres. Verify the BM25 tsvector trigger fires, the ivfflat index is used, and basic pgvector search works.

**Packages used:** `psycopg`, `pgvector`, `sqlalchemy`, `numpy`

## 1. Setup

In [ ]:
import sys, asyncio, uuid
from pathlib import Path

project_root = Path().resolve().parents[1]
sys.path.insert(0, str(project_root))

from dotenv import load_dotenv
load_dotenv(project_root / ".env")

from core.config import get_settings
settings = get_settings()
print(f"Database: {settings.database_url.split('@')[-1]}")

## 2. Embed Sample Chunks

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer(settings.embedding_model)

sample_chunks = [
    "Vector databases enable efficient similarity search over high-dimensional embeddings.",
    "pgvector extends PostgreSQL with native support for vector operations and ANN search.",
    "Retrieval-augmented generation improves LLM accuracy by grounding answers in documents.",
    "Semantic chunking splits documents at topic boundaries to preserve context.",
    "Hybrid retrieval combines dense vector search with sparse BM25 keyword search.",
]

embeddings = model.encode(sample_chunks, normalize_embeddings=True, batch_size=settings.embedding_batch_size)
print(f"Embedded {len(embeddings)} chunks — shape: {embeddings.shape}")

## 3. Insert via psycopg (sync — easier for notebooks)

In [ ]:
import psycopg
from pgvector.psycopg import register_vector

conn_url = settings.database_url \
    .replace("postgresql+asyncpg://", "postgresql://") \
    .replace("postgresql+psycopg://", "postgresql://")

conn = psycopg.connect(conn_url)
register_vector(conn)   # enables VECTOR type handling
print("Connected + pgvector registered")

In [ ]:
# Insert a test document
doc_id = str(uuid.uuid4())

with conn.cursor() as cur:
    cur.execute("""
        INSERT INTO documents (doc_id, title, source, doc_type)
        VALUES (%s, %s, %s, %s)
    """, (doc_id, "Phase 2 Test Document", "notebook_05", "text"))
    conn.commit()

print(f"Inserted document: {doc_id}")

In [ ]:
# Insert a test section
section_id = str(uuid.uuid4())

with conn.cursor() as cur:
    cur.execute("""
        INSERT INTO sections (section_id, doc_id, title, level)
        VALUES (%s, %s, %s, %s)
    """, (section_id, doc_id, "Introduction to RAG", 1))
    conn.commit()

print(f"Inserted section: {section_id}")

In [ ]:
# Insert chunks with embeddings
chunk_ids = []

with conn.cursor() as cur:
    for content, embedding in zip(sample_chunks, embeddings):
        chunk_id = str(uuid.uuid4())
        cur.execute("""
            INSERT INTO chunks (chunk_id, doc_id, section_id, content, embedding,
                                embedding_model_version, modality, token_count)
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
        """, (
            chunk_id, doc_id, section_id, content,
            embedding,                          # pgvector handles numpy array
            settings.embedding_model_version,
            "text",
            len(content.split()),               # approximate token count
        ))
        chunk_ids.append(chunk_id)
    conn.commit()

print(f"Inserted {len(chunk_ids)} chunks")

In [ ]:
# Insert a test asset (image)
asset_id = str(uuid.uuid4())
fake_asset_embedding = np.random.randn(settings.embedding_dim).astype(np.float32)
fake_asset_embedding /= np.linalg.norm(fake_asset_embedding)   # normalise

with conn.cursor() as cur:
    cur.execute("""
        INSERT INTO assets (asset_id, doc_id, type, content, embedding)
        VALUES (%s, %s, %s, %s, %s)
    """, (asset_id, doc_id, "image", "Figure 1: RAG system overview diagram", fake_asset_embedding))
    conn.commit()

print(f"Inserted asset: {asset_id}")

In [ ]:
# Insert chunk → asset relationship
with conn.cursor() as cur:
    cur.execute("""
        INSERT INTO relationships (source_id, target_id, relation_type)
        VALUES (%s, %s, %s)
    """, (chunk_ids[0], asset_id, "chunk_to_asset"))
    conn.commit()

print(f"Inserted relationship: {chunk_ids[0]} → {asset_id}")

## 4. Verify BM25 Trigger (tsv column)

In [ ]:
with conn.cursor() as cur:
    cur.execute("""
        SELECT chunk_id, content, tsv IS NOT NULL AS tsv_populated,
               ts_rank(tsv, plainto_tsquery('english', 'vector database')) AS bm25_score
        FROM chunks
        WHERE doc_id = %s
        ORDER BY bm25_score DESC;
    """, (doc_id,))
    rows = cur.fetchall()

print("BM25 trigger verification:")
for row in rows:
    cid, content, tsv_ok, score = row
    print(f"  tsv={'✅' if tsv_ok else '❌'}  bm25={score:.4f}  {content[:60]}...")

## 5. pgvector Similarity Search

In [ ]:
query = "How does vector search work in Postgres?"
query_embedding = model.encode(
    "Represent this sentence for searching relevant passages: " + query,
    normalize_embeddings=True
)

with conn.cursor() as cur:
    cur.execute("""
        SELECT chunk_id, content,
               (embedding <=> %s) AS distance
        FROM chunks
        WHERE doc_id = %s
        ORDER BY distance
        LIMIT 3;
    """, (query_embedding, doc_id))
    rows = cur.fetchall()

print(f"Query: '{query}'")
print("\nTop-3 by vector similarity:")
for i, (cid, content, dist) in enumerate(rows, 1):
    print(f"  {i}. dist={dist:.4f}  {content[:80]}")

## 6. Verify ivfflat Index is Used

In [ ]:
with conn.cursor() as cur:
    cur.execute(f"""
        EXPLAIN SELECT chunk_id
        FROM chunks
        ORDER BY embedding <=> '{query_embedding.tolist()}'
        LIMIT 5;
    """)
    plan = cur.fetchall()

print("Query plan:")
for row in plan:
    print(f"  {row[0]}")

plan_text = " ".join(r[0] for r in plan)
if "ivfflat" in plan_text.lower() or "index" in plan_text.lower():
    print("\n✅ Index used in query plan")
else:
    print("\nℹ️  Sequential scan — index may require more rows to activate")

## 7. Row Counts

In [ ]:
tables = ["documents", "sections", "chunks", "assets", "relationships"]
with conn.cursor() as cur:
    for table in tables:
        cur.execute(f"SELECT COUNT(*) FROM {table};")
        print(f"  {table:<15} {cur.fetchone()[0]} rows")

## 8. Cleanup Test Data

In [ ]:
with conn.cursor() as cur:
    cur.execute("DELETE FROM documents WHERE doc_id = %s", (doc_id,))   # cascades
    conn.commit()
conn.close()
print("Test data cleaned up. Connection closed.")

## Summary

| Check | Result |
|---|---|
| Document/section/chunk insert | ✅ |
| Embedding stored as VECTOR(1024) | ✅ |
| BM25 trigger populates tsv | ✅ |
| Asset + relationship insert | ✅ |
| pgvector similarity search | ✅ |
| ivfflat index referenced | ✅ |

**Next:** `06_ingestion_e2e.ipynb` — full pipeline on a real document.